# 4장 밑바닥부터 GPT 모델 구현하기: 따라 치기 노트북

『밑바닥부터 만들면서 배우는 LLM』 4장을 따라 치며 공부하는 노트북입니다. 2·3장과 같은 방식이에요.

1. 절마다 책을 보며 `✏️` 셀에 코드를 직접 입력하고 실행하세요. 셀은 필요한 만큼 추가하면 됩니다.
2. 주석은 **"이 줄이 왜 필요한지"를 내 말로** 쓰세요.
3. 🔍 **확인 셀**에서 `✅`가 나오면 다음 절로 넘어가세요.

**확인 셀은 3장처럼 숫자까지 책과 비교해요.** 책과 같은 위치에서 `torch.manual_seed(123)`을 호출해야 같은 숫자가 나와요.

**3장과 연결**: 4장의 트랜스포머 블록은 3장에서 다시 짠 `MultiHeadAttention`을 그대로 씁니다. 아래 0번 셀에서 내 저장소의 `ch03_attention/attention.py`를 불러와요.

막히면 [책의 공식 코드](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch04/01_main-chapter-code)를 참고하세요.

## 0. 환경 설정
`tiktoken`(2장의 BPE 토크나이저)과 3장의 `attention.py`를 준비합니다. Colab이 아니라 저장소 폴더에서 실행한다면 `wget` 줄은 건너뛰고 `sys.path`만 맞추면 돼요.

In [ ]:
!pip install -q tiktoken
!wget -q -O attention.py https://raw.githubusercontent.com/uisang2004-maker/LLM-/main/ch03_attention/attention.py

In [ ]:
import torch
import torch.nn as nn
import tiktoken
from attention import MultiHeadAttention  # 3장에서 보지 않고 다시 짠 클래스

print("torch version:", torch.__version__)
print("tiktoken version:", tiktoken.__version__)


def _close(actual, expected, atol=1e-3):
    """확인 셀용: 책의 출력값과 소수점 셋째 자리까지 같은지"""
    return torch.allclose(torch.as_tensor(actual).detach().float(), torch.tensor(expected), atol=atol)

## 4.1 LLM 구조 구현하기
GPT-2 가장 작은 모델(1억 2,400만 파라미터)의 설정 `GPT_CONFIG_124M`을 만들고, 속은 비어 있는 `DummyGPTModel`로 **전체 뼈대**부터 잡습니다. 입력은 2장의 tiktoken으로 토큰화한 문장 두 개예요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: GPT_CONFIG_124M 딕셔너리 (vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate, qkv_bias)
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # 어휘 사전 크기: GPT-2 BPE 토크나이저(tiktoken)의 토큰 개수
    "context_length": 1024,  # 한 번에 볼 수 있는 최대 토큰 수 (위치 임베딩 개수)
    "emb_dim": 768,          # 토큰 하나를 나타내는 벡터의 길이
    "n_heads": 12,           # 멀티 헤드 어텐션의 헤드 수
    "n_layers": 12,          # 트랜스포머 블록을 몇 개 쌓을지
    "drop_rate": 0.1,        # 드롭아웃 비율 (10%를 무작위로 0으로)
    "qkv_bias": False,       # 쿼리·키·밸류 Linear 층에 편향을 쓸지
}

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: DummyGPTModel, DummyTransformerBlock, DummyLayerNorm 클래스
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])      # 토큰 ID → 벡터
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])  # 위치 번호 → 벡터
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(                                     # 트랜스포머 블록 12개를 차례로 통과
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)  # 벡터 → 어휘 사전 전체의 점수(로짓)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)                                         # (b, seq_len, emb_dim)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # (seq_len, emb_dim)
        x = tok_embeds + pos_embeds   # 2장처럼 토큰 임베딩 + 위치 임베딩 (브로드캐스팅으로 배치마다 더함)
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)     # (b, seq_len, vocab_size)
        return logits


class DummyTransformerBlock(nn.Module):  # 4.5절에서 진짜 TransformerBlock으로 바꿀 자리
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x  # 아무것도 하지 않고 입력을 그대로 돌려줌


class DummyLayerNorm(nn.Module):  # 4.2절에서 진짜 LayerNorm으로 바꿀 자리
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: tokenizer, batch ("Every effort moves you", "Every day holds a"를 torch.stack)
tokenizer = tiktoken.get_encoding("gpt2")  # 2장에서 쓴 GPT-2 BPE 토크나이저

batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))  # 문장 → 토큰 ID 4개
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)  # (4,) 두 개를 쌓아서 (2, 4) 배치로
print(batch)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.manual_seed(123), model = DummyGPTModel(GPT_CONFIG_124M), logits
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("출력 크기:", logits.shape)  # (배치 2, 토큰 4, 어휘 사전 50257): 토큰마다 다음 토큰 후보 전체의 점수
print(logits)

In [ ]:
# 🔍 확인
print(batch)
print("logits.shape:", logits.shape)
assert batch.tolist() == [[6109, 3626, 6100, 345], [6109, 1110, 6622, 257]], "토큰 ID가 책과 달라요. tiktoken.get_encoding(\"gpt2\")와 문장 철자를 확인"
assert GPT_CONFIG_124M["emb_dim"] == 768 and GPT_CONFIG_124M["n_layers"] == 12 and GPT_CONFIG_124M["n_heads"] == 12, "설정값이 책과 달라요"
assert tuple(logits.shape) == (2, 4, 50257), "(배치 2, 토큰 4, 어휘 사전 크기 50257)이어야 해요. out_head의 출력 크기 확인"
print("✅ GPT 뼈대 확인: 토큰마다 어휘 사전 전체에 대한 점수(로짓)가 나옴")

## 4.2 층 정규화로 활성화 정규화하기
층의 출력을 **평균 0, 분산 1**로 맞춰 학습을 안정시킵니다. 먼저 작은 예시(`batch_example`)로 직접 계산해 보고, `LayerNorm` 클래스로 묶어요. `scale`, `shift`는 모델이 정규화 결과를 다시 조정할 수 있게 하는 학습 파라미터예요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.manual_seed(123), batch_example = torch.randn(2, 5), Linear(5, 6) + ReLU 층, out의 mean·var

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: (out - mean) / torch.sqrt(var)로 직접 정규화한 out_norm

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: LayerNorm 클래스 (eps, scale, shift, unbiased=False), ln, out_ln

In [ ]:
# 🔍 확인
print("정규화 전 mean:", mean.squeeze(), " var:", var.squeeze())
print("LayerNorm 뒤 mean:", out_ln.mean(dim=-1), " var:", out_ln.var(dim=-1, unbiased=False))
assert _close(mean, [[0.1324], [0.2170]]) and _close(var, [[0.0231], [0.0398]]), "정규화 전 평균·분산이 책과 달라요. torch.manual_seed(123) 위치와 dim=-1 확인"
assert _close(out_norm.mean(dim=-1), [0.0, 0.0]) and _close(out_norm.var(dim=-1), [1.0, 1.0]), "직접 정규화한 결과가 평균 0, 분산 1이 아니에요"
assert tuple(out_ln.shape) == (2, 5), "LayerNorm은 모양을 바꾸지 않아요"
assert _close(out_ln.mean(dim=-1), [0.0, 0.0]) and _close(out_ln.var(dim=-1, unbiased=False), [1.0, 1.0], atol=1e-2), "LayerNorm 결과가 평균 0, 분산 1이 아니에요. unbiased=False와 eps 확인"
assert sum(p.numel() for p in ln.parameters()) == 10, "scale, shift가 emb_dim 크기의 nn.Parameter여야 해요 (5 + 5)"
print("✅ 층 정규화 확인")

## 4.3 GELU 활성화 함수를 사용하는 피드 포워드 네트워크 구현하기
ReLU 대신 쓰는 부드러운 활성화 함수 **GELU**(tanh 근사식)를 만들고, `emb_dim → 4·emb_dim → emb_dim`으로 넓혔다 좁히는 `FeedForward`를 만듭니다. 그래프로 ReLU와 비교해 보세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: GELU 클래스 (0.5 * x * (1 + tanh(√(2/π) * (x + 0.044715 * x³))))

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: GELU와 ReLU 그래프 그리기 (matplotlib)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: FeedForward 클래스, ffn = FeedForward(GPT_CONFIG_124M), x = torch.rand(2, 3, 768), out

In [ ]:
# 🔍 확인
_g = GELU()(torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0]))
print("GELU:", _g)
assert _close(_g, [-0.0036, -0.1588, 0.0, 0.8412, 2.9964]), "GELU 값이 달라요. tanh 근사식의 상수(√(2/π), 0.044715)와 x³ 확인"
assert tuple(out.shape) == (2, 3, 768), "피드 포워드는 입력과 출력 모양이 같아야 해요"
_sizes = [tuple(m.weight.shape) for m in ffn.modules() if isinstance(m, nn.Linear)]
assert _sizes == [(3072, 768), (768, 3072)], "첫 Linear는 768 → 3072(4배), 두 번째는 3072 → 768이어야 해요"
print("✅ GELU, 피드 포워드 확인")

## 4.4 숏컷 연결 추가하기
층이 깊어지면 앞쪽 층의 그레디언트가 작아지는 문제(그레디언트 소실)를, **층의 입력을 출력에 더하는** 숏컷(잔차) 연결로 막습니다. 숏컷이 있을 때와 없을 때 층마다 그레디언트 크기를 비교해 보세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: ExampleDeepNeuralNetwork 클래스 (use_shortcut), print_gradients 함수

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: layer_sizes, sample_input, torch.manual_seed(123) 후 model_without_shortcut으로 print_gradients

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.manual_seed(123) 후 model_with_shortcut으로 print_gradients

In [ ]:
# 🔍 확인: 첫 층(layers.0)과 마지막 층(layers.4)의 그레디언트 크기
def _grad_means(m):
    return [p.grad.abs().mean().item() for n, p in m.named_parameters() if "weight" in n]
_no, _yes = _grad_means(model_without_shortcut), _grad_means(model_with_shortcut)
print("숏컷 없음:", [round(g, 6) for g in _no])
print("숏컷 있음:", [round(g, 6) for g in _yes])
assert _close([_no[0], _no[4]], [0.0002, 0.0050], atol=1e-4), "숏컷 없는 모델의 그레디언트가 책과 달라요. torch.manual_seed(123) 위치, layer_sizes, GELU 확인"
assert _close([_yes[0], _yes[4]], [0.2217, 1.3259]), "숏컷 있는 모델의 그레디언트가 책과 달라요. x = x + layer_output 확인"
assert _yes[0] > 100 * _no[0], "숏컷이 있으면 첫 층의 그레디언트가 훨씬 커야 해요"
print("✅ 숏컷 연결: 앞쪽 층까지 그레디언트가 살아 있음")

## 4.5 어텐션과 선형 층을 트랜스포머 블록에 연결하기
지금까지 만든 부품을 조립합니다: **층 정규화 → 멀티 헤드 어텐션 → 드롭아웃 → 숏컷**, 그다음 **층 정규화 → 피드 포워드 → 드롭아웃 → 숏컷**. GPT-2는 정규화를 어텐션 *앞*에 두는 Pre-LayerNorm 구조예요. `MultiHeadAttention`은 0번 셀에서 불러온 3장의 클래스를 씁니다.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: TransformerBlock 클래스 (att, ff, norm1, norm2, drop_shortcut)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.manual_seed(123), x = torch.rand(2, 4, 768), block, output

In [ ]:
# 🔍 확인
print("Input shape:", x.shape)
print("Output shape:", output.shape)
assert tuple(output.shape) == (2, 4, 768), "트랜스포머 블록은 입력과 출력 모양이 같아야 해요 (그래야 12개를 쌓을 수 있음)"
_n = sum(p.numel() for p in block.parameters())
print("블록 하나의 파라미터 수:", f"{_n:,}")
assert _n == 7_085_568, "블록 하나의 파라미터 수가 달라요. 어텐션(qkv_bias=False, out_proj는 bias 있음), 피드 포워드(4배), LayerNorm 2개 확인"
block.eval()
_y = block(x)
_x2 = x.clone(); _x2[:, 2:, :] = 0.0
assert torch.allclose(_y[:, :2], block(_x2)[:, :2], atol=1e-6), "블록 안에서도 앞 토큰은 뒤 토큰을 보면 안 돼요 (어텐션 마스크 확인)"
print("✅ 트랜스포머 블록 확인")

## 4.6 GPT 모델 만들기 ⭐
4장의 최종 목표예요. `DummyGPTModel`의 빈 부품을 진짜 `TransformerBlock`과 `LayerNorm`으로 바꿉니다. 파라미터 수와 모델 크기(MB)도 직접 계산해 보세요. **파라미터가 1억 6,300만 개인데 왜 "124M"이라 부르는지**(가중치 묶기)를 눈여겨보세요.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: GPTModel 클래스 (tok_emb, pos_emb, drop_emb, trf_blocks, final_norm, out_head)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: torch.manual_seed(123), model = GPTModel(GPT_CONFIG_124M), out = model(batch)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: total_params, 가중치 묶기를 고려한 파라미터 수, total_size_mb (float32 = 4바이트)

In [ ]:
# 🔍 확인
print("Output shape:", out.shape)
print(f"Total number of parameters: {total_params:,}")
print(f"가중치 묶기 고려: {total_params_gpt2:,}")
print(f"Total size of the model: {total_size_mb:.2f} MB")
assert tuple(out.shape) == (2, 4, 50257), "(배치 2, 토큰 4, 어휘 사전 50257)이어야 해요"
assert total_params == 163_009_536, "전체 파라미터 수가 책(163,009,536)과 달라요. 블록 12개, out_head의 bias=False 확인"
assert tuple(model.tok_emb.weight.shape) == tuple(model.out_head.weight.shape) == (50257, 768), "토큰 임베딩과 출력 층의 가중치 모양이 같아야 가중치 묶기가 가능해요"
assert total_params_gpt2 == 124_412_160, "출력 층을 뺀 파라미터 수는 124,412,160이어야 해요"
assert abs(total_size_mb - 621.83) < 0.01, "모델 크기는 파라미터 수 × 4바이트를 MB(1024²)로 나눈 621.83이어야 해요"
print("✅ GPT 모델 확인")

## 4.7 텍스트 생성하기
모델이 내놓은 **마지막 토큰의 로짓**에서 가장 큰 값(`argmax`)을 골라 입력 뒤에 붙이는 것을 반복합니다. 아직 학습하지 않은 모델이라 **이상한 문장이 나오는 게 정상**이에요. 5장에서 학습시킵니다.

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: generate_text_simple 함수 (idx_cond 자르기, logits[:, -1, :], softmax, argmax, torch.cat)

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: start_context = "Hello, I am", encoded_tensor (unsqueeze(0))

In [ ]:
# ✏️ 책을 보며 직접 입력하세요: model.eval(), out = generate_text_simple(..., max_new_tokens=6, context_size=GPT_CONFIG_124M["context_length"]), decoded_text

In [ ]:
# 🔍 확인
print("Output:", out)
print("Output length:", len(out[0]))
print(decoded_text)
assert encoded_tensor.tolist() == [[15496, 11, 314, 716]], "\"Hello, I am\"의 토큰 ID가 달라요. unsqueeze(0)로 배치 차원을 붙였는지 확인"
assert tuple(out.shape) == (1, 10), "입력 4개 + 새 토큰 6개 = 10개여야 해요"
assert out[0, :4].tolist() == [15496, 11, 314, 716], "생성 결과의 앞부분은 입력 그대로여야 해요 (torch.cat 순서 확인)"
_again = generate_text_simple(model, encoded_tensor, 6, GPT_CONFIG_124M["context_length"])
assert torch.equal(out, _again), "eval 모드에서 argmax로 고르면 매번 같은 결과가 나와야 해요. model.eval()을 했는지 확인"
print("✅ 텍스트 생성 확인 (학습 전이라 뜻 없는 문장이 정상)")
if out[0].tolist() == [15496, 11, 314, 716, 27018, 24086, 47843, 30961, 42348, 7267]:
    print("   책과 같은 토큰까지 나왔어요: 모든 층의 생성 순서가 책과 같다는 뜻")
else:
    print("   생성된 토큰이 책과 달라도 괜찮아요. 3장의 attention.py에서 층을 만든 순서가 책과 다르면 초기 가중치가 달라져요")

---
## 4장 마무리
- [ ] `notes.md` 채우기
- [ ] 이 노트북을 닫고 `LayerNorm`, `FeedForward`(+`GELU`), `TransformerBlock`을 보지 않고 다시 짜기
- [ ] `GPTModel`과 `generate_text_simple`을 보지 않고 다시 짜기 (텐서 모양을 주석으로 적으며). 다시 짠 코드는 `gpt.py`로 저장해 5장에서 `import`해 보기
- [ ] 노트북을 .ipynb로 다운로드해 이 폴더에 넣고 커밋, push